# Assignment 1. Machine Learning and Deep Learning
#### INSERT YOUR NAME AND ID HERE

This assignment consists of two parts:

**Part 1: Student Performance Prediction**
- We will build models to predict student performance using the "Student Performance Dataset", which includes student grades, demographic, social and school related features collected through school reports and questionnaires.
- You can read more about this dataset at [the UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/320/student+performance).
- In the first section, you will write a linear regression model to predict students' grades at the end of the year.
- In the second section, you will build a classifier to predict if a student's performance will be above average.

**Part 2: Music Century Classification**
- We will construct models to predict the century in which a music piece was released using the "YearPredictionMSD Data Set", derived from the Million Song Dataset.
- Make sure you download the version of the dataset from the moodle and not from UCI.
- Relevant links:
  - https://archive.ics.uci.edu/ml/datasets/yearpredictionmsd
  - http://millionsongdataset.com/pages/tasks-demos/#yearrecognition
- You will implement logistic regression from scratch using gradient descent.
- You will then see how the same task can be accomplished using PyTorch.

## Files Structure

The assignment includes:
1. This notebook file (ML_DL_Assignment1.ipynb)
2. The student performance dataset file (student-mat.csv)
3. The year prediction dataset file (YearPredictionMSD.csv)

You will mount and load the datasets from Google Drive. Make sure you have both dataset files in the same directory in your Google Drive.

## Submission Requirements

When you are finished with the assignment, submit This notebook file, including your code and presented results. Make sure all results are visible and all code cells have been executed in order.

## Important Note

In **Part 2** (Music Century Classification), until Section 2.8, you are **not allowed to import additional packages (especially not PyTorch)**. One of the objectives is to understand how the training procedure actually operates before working with PyTorch's autograd engine. Importing the PyTorch package before Section 2.8 will deduct from your points.

In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt
import sys

---
# Part 1. Student Performance Prediction

## 1. The Data

### 1.1 Load your dataset from Google Drive

Start by setting up a Google Colab notebook in which to do your work.
You might find this link helpful:

- https://colab.research.google.com/github/googlecolab/colabtools/blob/master/notebooks/colab-github-demo.ipynb

To process and read the data, we use the popular `pandas` package for data analysis.

Now that your notebook is set up, we can load the data into the notebook. You will need to upload the "student-mat.csv" dataset to a directory in Google Drive and mount your Google Drive to the Colab notebook.

Here are some resources to help you get started:

[http://colab.research.google.com/notebooks/io.ipynb](https://colab.research.google.com/notebooks/io.ipynb)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')  # Your Google Drive will be mounted to /content/gdrive

drive_path = '/content/gdrive/My Drive/Intro_to_Deep_Learning/Assignment1/' # TODO - UPDATE ME WITH THE TRUE PATH!

print("looking for dataset in: "+drive_path+'student-mat.csv')
df = pandas.read_csv(drive_path+'student-mat.csv')
df = df.sample(frac=1) # This line scrambles the order of the dataframe

Now that the data is loaded to your Colab notebook, you should be able to display the Pandas DataFrame `df` as a table:

In [ ]:
df

Notice that the data consists of multiple types. There are numeric columns such as 'age','freetime' and 'G3' but there are also a few categorical columns for example 'school','address','Mjob' etc. Since we want to use a numeric estimator we need all our data to be numeric. We can use indicators to translate the categories into numeric values. In the pandas library it is done using the [get_dummies](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) function.

Notice that in this assignment we will not be using any multiple category columns (to simplify the task a bit) so we remove the columns 'Mjob', 'Fjob', 'reason' and 'guardian'.

In [ ]:
df_no_multi_categorical = df.drop(["Mjob","Fjob","reason","guardian"],axis=1)
df_numerical = pandas.get_dummies(df_no_multi_categorical, drop_first=True)

Notice how each categorical column has been converted to a binary numerical value.

In [ ]:
df_numerical

### 1.2 Prepare the data

Splitting a dataset into training and testing sets is a common practice in machine learning. The primary reason for doing so is to evaluate the performance of a model on unseen data. By splitting the dataset into training and testing sets, we can train our model on the training set and evaluate its performance on the testing set.

In this assignment the code used to prepare the data is already given to you

In [ ]:
df_train = df_numerical[:270]
df_test = df_numerical[270:]

It can also be beneficial to **normalize** the columns, so that each column (feature) has the *same* mean and standard deviation.

In [ ]:
feature_means = df_train.mean().to_numpy()
feature_stds  = df_train.std().to_numpy()
feature_stds[feature_stds==0] = 0.01
train_norm = (df_train - feature_means) / feature_stds
test_norm = (df_test - feature_means) / feature_stds

#### Food for thought:
*Notice how in our code, we normalized the test set using the training data means and standard deviations. This is not a bug. Why would it be improper to compute and use test set means and standard deviations? (Hint: Remember what we want to use the test accuracy to measure.)*

Finally we split the labels and the features of the dataset into separate matrices.

In [ ]:
# convert to numpy
train_x = train_norm.drop("G3",axis=1).to_numpy()
train_s = train_norm["G3"].to_numpy()
test_x = test_norm.drop("G3",axis=1).to_numpy()
test_s = test_norm["G3"].to_numpy()

### 1.3 Check for correlation between the variables in the input

Before you start fitting models to the data it is important to understand it. Calculate and show (in whatever way fits you) the correlation between each of the input parameters ('age','Medu','Fedu','traveltime', etc) and the predicted parameter 'G3'.

You can use the [seaborn](https://seaborn.pydata.org/) library to aid you in this. This section is only for educational purposes, make sure you do not change the shape or values of the training or test data in it.

In [ ]:
import seaborn as sn
# Insert your code here

#### Food for thought:
*What can you learn from this test? Are all input parameters equally important? Which ones could you omit? Which two variables are the best predictor of student performance at the end of the year and why?*

## 2. Linear Regression

The first task you are going to tackle is Linear regression. In this task we will predict a student's performance according to their different parameters.

### 2.1 Fit a linear model

You can use the LinearRegression class from sklearn.linear_model. You can read more on it at:

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()

# Insert your code here

### 2.2 Test the accuracy of the model

There are several ways to measure the accuracy of a linear regression model. Test your model using the '[predict](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)' function on the test set. You can measure the accuracy using MSE:

$MSE= \frac 1N\sum(s-\hat{s})^2$

or using the $R^2$ value:

$1-\frac{\text{unexplained variance=MSE}}{\text{Total Variation}}=1-\frac{\sum(s-\hat{s})^2}{\sum(s-\bar{s})^2}$

You can read on it further [here](https://en.wikipedia.org/wiki/Coefficient_of_determination).

In [ ]:
# Insert your code here

If everything went correctly you should get an MSE value of around 0.11 and around 0.87 for $R^2$.

### 2.3 Save the parameters of the model

Run the code block below to see your model's coefficients and intercept. These values will be used for grading, so make sure they are correct.

In [ ]:
coef = reg.coef_
intrcpt = reg.intercept_
print("coefficients: ")
print(coef)
print("intercept: ")
print(intrcpt)

### 2.4 Test your skills in Linear algebra and compare the result to sklearn

While using Sklearn's built in linear regression is easy it is also important to see that it is not too complicated to write the regression by yourself. In class you were taught that the linear regression could be estimated using the Least Squares method. The Pseudo-Inverse is defined as:

$\theta^* = (X^TX)^{-1}X^Ts$

Use this knowledge to write the linear regression yourself. Implement the LeastSquares function in the cell below.

In this function you are not allowed to use the sklearn or scipy libraries.

In [ ]:
def LeastSquares(X, y):
  '''
    Calculates the Least squares solution to the problem X*theta=y using the least squares method
    :param X: numpy input matrix, size [N,m+1] (feature 0 is a column of 1 for bias)
    :param y: numpy input vector, size [N]
    :return theta = (Xt*X)^(-1) * Xt * y: numpy output vector, size [m+1]
    N is the number of samples and m is the number of features=28
  '''
  # Insert your code here
  return ...

In [ ]:
import scipy

from copy import deepcopy
train_x_tag = np.hstack([np.ones([train_x.shape[0],1]),train_x]) # we add the bias term directly to the linear equation
theta = LeastSquares(train_x_tag,train_s)

Test the accuracy of the model, as in 2.2

In [ ]:
# Insert your code here

## 3. Linear Classification

In this section you will be classifying whether a student's performance in the end of the year was above average or not according to all other metrics.

### 3.1 Prepare the data: split the dataset between students with grades above average and below and then split again for train test

To set up our data for classification, we'll use the "G3" field to represent whether a student achieved a performance above average or below. In our case `df_cl["G3"]` will be 1 if the student's performance is above average and 0 otherwise.

In [ ]:
df_cl = deepcopy(df_numerical)
print("average performance= "+ str(df_cl['G3'].mean()))
df_cl["G3"] = df["G3"].map(lambda x: int(x > 10))

Next you need to normalize and split the dataframe into a training and test sets just like was done in part 1.2.

**Notice that when normalizing you do not normalize the labels**

In [ ]:
# Insert your code here for normalizing and splitting the dataset for train and test
train_cl_x = ...
train_cl_s = ...
test_cl_x = ...
test_cl_s = ...

### 3.2 Fit an SVM model to the data to predict whether a grade is above or below average

Read on [the svm implementation of sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html). Fit the model on the training classification dataset you built.

In [ ]:
import sklearn.svm
svc = sklearn.svm.LinearSVC()

# Insert your code here:

### 3.3 Test your model

For classification accuracy is usually calculated as the number of correct predictions divided by number of total predictions:

$$\frac{\text{Number of correct predictions}}{\text{Number of total predictions}}$$

The accuracy is calculated on the test set. Implement the accuracy function in the cell below and run the test after it:

In [ ]:
def classification_accuracy(model, X, s):
  '''
    calculate the accuracy for the classification problem
    :param model: the classification model class
    :param X: numpy input matrix, size [N,m]
    :param s: numpy input vector of ground truth labels, size [N]
    :return: accuracy of the model = (correct classifications)/(total classifications) type float
    N is the number of samples and m is the number of features=28
  '''
  # Insert your code here
  return ...

In [ ]:
classification_accuracy(svc,test_cl_x,test_cl_s)

If everything worked correctly you should get an accuracy of around 90%

### 3.4 Save the parameters of the model

Run the code block below to see your model's coefficients, intercept, and classes. These values will be used for grading, so make sure they are correct.

In [ ]:
coef_cl = svc.coef_
intrcpt_cl = svc.intercept_
classes_cl = svc.classes_
print(coef_cl)
print(intrcpt_cl)
print(classes_cl)

---
# Part 2. Music Century Classification

## 1. Data

Make sure you have the `YearPredictionMSD.csv` dataset from the moodle in the same directory in your Google Drive as you set earlier.

In [ ]:
csv_path = drive_path + 'YearPredictionMSD.csv'
t_label = ["year"]
x_labels = ["var%d" % i for i in range(1, 91)]
df2 = pandas.read_csv(csv_path)

Now that the data is loaded to your Colab notebook, you should be able to display the Pandas DataFrame `df2` as a table:

In [ ]:
df2

To set up our data for classification, we'll use the "year" field to represent whether a song was released in the 21st century. In our case `df2["year"]` will be 1 if the year was released after 2000, and 0 otherwise.

In [ ]:
df2["year"] = df2["year"].map(lambda x: int(x > 2000))

In [ ]:
df2.head(20)

### 1.1 - Train Test Split

The data set description text asks us to respect the below train/test split to avoid the "producer effect". That is, we want to make sure that no song from a single artist ends up in both the training and test set.

#### Food for thought:
Why would it be problematic to have some songs from an artist in the training set, and other songs from the same artist in the test set? (Hint: Remember that we want our test accuracy to predict how well the model will perform in practice on a song it hasn't learned about.)

In [ ]:
# train test split
df_train = df2[:463715]
df_test = df2[463715:]

# convert to numpy
train_xs = df_train[x_labels].to_numpy()
train_ts = df_train[t_label].to_numpy()
test_xs = df_test[x_labels].to_numpy()
test_ts = df_test[t_label].to_numpy()

### 1.2 Normalizing the Dataset

Normalize the data by subtracting the mean and dividing by the std just like in Part 1.

In [ ]:
# Insert your code here:
train_norm_xs = ...
test_norm_xs = ...

### 1.3 Splitting the Dataset

Finally, we'll move some of the data in our training set into a validation set.

#### Food for thought:
Why should we limit how many times we use the test set, and how do we use the validation set during the model building process?

In [ ]:
# shuffle the training set
reindex = np.random.permutation(len(train_xs))
train_xs = train_xs[reindex]
train_norm_xs = train_norm_xs[reindex]
train_ts = train_ts[reindex]

# use the first 50000 elements of `train_xs` as the validation set
train_xs, val_xs           = train_xs[50000:], train_xs[:50000]
train_norm_xs, val_norm_xs = train_norm_xs[50000:], train_norm_xs[:50000]
train_ts, val_ts           = train_ts[50000:], train_ts[:50000]

## 2. Classification

We will now build a *classification* model to perform decade classification. We have written a few helper functions for you below (`sigmoid`, `cross_entropy` and `get_accuracy`). All other code that you write in this section should be vectorized whenever possible (i.e., avoid unnecessary loops). Feel free to add more testing to the notebook to validate your code.

In [ ]:
def sigmoid(z):
  return 1 / (1 + np.exp(-z))
    
def cross_entropy(t, y):
  return -t * np.log(y) - (1 - t) * np.log(1 - y)

def get_accuracy(y, t):
  acc = 0
  N = 0
  for i in range(len(y)):
    N += 1
    if (y[i] >= 0.5 and t[i] == 1) or (y[i] < 0.5 and t[i] == 0):
      acc += 1
  return acc / N

### 2.1 Prediction

Implement the function `pred` in the cell below that computes the prediction `y` based on logistic regression, i.e., a single layer with weights `w` and bias `b`. The output is given by:

\begin{equation}
y = \sigma({\bf w}^T {\bf x} + b),
\end{equation}

where the value of $y$ is an estimate of the probability that the song is released in the current century, namely ${\rm year} =1$.

In [ ]:
def pred(w, b, X):
  """
  Returns the prediction `y` of the target based on the weights `w` and scalar bias `b`.

  Preconditions: np.shape(w) == (90,)
                 type(b) == float
                 np.shape(X) = (N, 90) for some N
  Postconditions: np.shape(y)==(N,)

  >>> pred(np.zeros(90), 1, np.ones([2, 90]))
  array([0.73105858, 0.73105858]) # It's okay if your output differs in the last decimals
  """
  # Insert your code here
  return ...

In [ ]:
pred(np.zeros(90), 1, np.ones([2, 90]))

### 2.2 Cost

Assuming the loss function is the cross entropy function, implement the cost(risk) function in the cell below which returns the mean of the loss function on all inputs.

$$\mathcal{L}_\mathcal{P}(\text{Cross Entropy}) = \mathbb{E}_{(y,t)\sim\mathcal{P}}\left\{\text{CE}(t,s)\right\}$$

In [ ]:
def cost(y, t):
  """
  Returns the cost(risk function) `L` of the prediction 'y' and the ground truth 't'.

  - parameter y: prediction
  - parameter t: ground truth
  - return L: cost/risk
  Preconditions: np.shape(y) == (N,) for some N
                 np.shape(t) == (N,)
  
  Postconditions: type(L) == float
  >>> cost(0.5*np.ones(90), np.ones(90))
  0.69314718 # It's okay if your output differs in the last decimals
  """
  # Insert your code here
  return ...

In [ ]:
print(cost(0.5*np.ones(4), np.ones(4)))

### 2.3 Derivative of the cost

Take a pen and paper and calculate the analytical derivative of the cost function with respect to the weights and bias. Use the formula calculated to implement the function `derivative_cost` in the cell below that computes and returns the gradients $\frac{\partial\mathcal{L}}{\partial {\bf w}}$ and $\frac{\partial\mathcal{L}}{\partial b}$. Here, `X` is the input, `y` is the prediction, and `t` is the true label.

In [ ]:
def derivative_cost(X, y, t):
  """
  Returns a tuple containing the gradients dLdw and dLdb.

  Precondition: np.shape(X) == (N, 90) for some N
                np.shape(y) == (N,)
                np.shape(t) == (N,)

  Postcondition: np.shape(dLdw) = (90,)
           type(dLdb) = float
           return dLdw,dldb
  """
  # Insert your code here
  dLdw = ...
  dLdb = ...
  return (dLdw, dLdb)

In [ ]:
dldw, dldb = derivative_cost(np.ones([10,90]), np.ones(10), np.ones(10))
print(dldw.shape)
print(type(dldb))

### 2.4 Derivative approximation

We can check that our derivative is implemented correctly using the finite difference rule. In 1D, the finite difference rule tells us that for small $h$, we should have

$$\frac{f(x+h) - f(x)}{h} \approx f'(x)$$

Make sure that $\frac{\partial\mathcal{L}}{\partial b}$ is implemented correctly by comparing the result from `derivative_cost` with the empirical cost derivative computed using the above numerical approximation.

In [ ]:
# Your code goes here

'''
r1 = ...
r2 = ...
print("The analytical results is -", r1)
print("The algorithm results is - ", r2)
'''

Make sure that $\frac{\partial\mathcal{L}}{\partial {\bf w}}$ is implemented correctly.

In [ ]:
# Your code goes here. You might find this below code helpful: but it's
# up to you to figure out how/why, and how to modify the code

'''
r1 = ...
r2 = ...
print("The analytical results is -", r1)
print("The algorithm results is - ", r2)
'''

### 2.5 Gradient descent

Now that you have a gradient function that works, we can actually run gradient descent. Complete the following code that will run stochastic gradient descent training:

In [ ]:
def run_gradient_descent(w0, b0, mu=0.1, batch_size=100, max_iters=100):
  """Return the values of (w, b) after running gradient descent for max_iters.
  We use:
    - train_norm_xs and train_ts as the training set
    - val_norm_xs and val_ts as the test set
    - mu as the learning rate
    - (w0, b0) as the initial values of (w, b)

  Precondition: np.shape(w0) == (90,)
                type(b0) == float

  Postcondition: np.shape(w) == (90,)
                 type(b) == float
  """
  w = w0
  b = b0
  iter = 0
  max_acc = 0
  opt_w = w
  opt_b = b
  cost_list = []
  acc_list  = []
  while iter < max_iters:
    # shuffle the training set (there is code above for how to do this)
    # <===

    for i in range(0, len(train_norm_xs), batch_size): # iterate over each minibatch
      # minibatch that we are working with:
      X = train_norm_xs[i:(i + batch_size)]
      t = train_ts[i:(i + batch_size), 0]

      # since len(train_norm_xs) does not divide batch_size evenly, we will skip over
      # the "last" minibatch
      if np.shape(X)[0] != batch_size:
        continue

      # compute the prediction
      # <===
      # calculate gradient(backpropegate)
      # <===
      # update w and b(step)
      # <===
      # increment the iteration count
      iter += 1
      # compute and print the *validation* loss and accuracy
      if (iter % 40 == 0):
        # <===
        val_cost = ...
        val_acc = ...
        cost_list.append(val_cost)
        acc_list.append(val_acc)
        # save the best weights and biases
        if val_acc>max_acc:
          opt_w = w
          opt_b = b

        print("Iter %d. [Val Acc %.0f%%, Loss %f]" % (
              iter, val_acc * 100, val_cost))

      if iter >= max_iters:
        break


  return opt_w, opt_b, cost_list, acc_list

### 2.6 Running everything!

Call `run_gradient_descent` with the weights and biases all initialized to zero. Test yourself with different $\mu$ values and show that if mu is too small then convergence is slow and if mu is too large then the optimization algorithm does not converge. You can add more automation and plot functions to help you find the best configuration.

In [ ]:
w0 = np.zeros(90)
b0 = np.zeros(1)[0]

# choose values
mu = ...
max_iters = ...
batch_size = ...


# Write your code here
# opt_w,opt_b,cost_list,acc_list = run_gradient_descent(w0,b0,mu,batch_size,max_iters)
# plt.plot(range(0,max_iters,40),acc_list,"r-")
# plt.title("classification accuracy per iteration; $\mu$="+str(mu)+" batch size="+str(batch_size))

### 2.7 Results

Using the optimal values of `w` and `b` from part 2.6, compute your training accuracy, validation accuracy, and test accuracy. Are there any differences between those three values? If so, why?

In [ ]:
# Use the optimal w and b from part 2.6
# Write your code here

train_acc = ...
val_acc = ...
test_acc = ...

print('train_acc = ', train_acc, ' val_acc = ', val_acc, ' test_acc = ', test_acc)

### 2.8 Using PyTorch

Writing a classifier like this is instructive, and helps you understand what happens when we train a model. However, in practice, we rarely write model building and training code from scratch. Instead, we typically use one of the well-tested libraries available in a package. The following example shows you how this task could have been achieved using the deep learning library, PyTorch. The library greatly simplifies the steps needed to create a learning model. Though there is nothing you need to complete in this section we suggest you read this section thoroughly and make sure you understand all the code. In the next assignment you will need to build a deep learning model yourself.

The first step required to use the PyTorch module is to create a class which will be our model. In this case we will use a linear layer with a custom size (in your assignment you used a 90,1 linear layer meaning an input size of 90 and an output size of 1). We also add a sigmoid function to restrict the values between 0 and 1.

The forward function is called every time you call the model by name. It is equivalent to the prediction function you wrote but it serves another purpose since it saves all the operations done to the tensor which can then be used to calculate the gradients.

In [ ]:
import torch
class single_layer(torch.nn.Module):
  def __init__(self,input_size,output_size):
    super(single_layer,self).__init__()
    self.neuron = torch.nn.Linear(input_size,output_size)
    self.sigmoid = torch.nn.Sigmoid()

  def forward(self,X):
    out = self.neuron(X)
    out = self.sigmoid(out)
    return out

We can now create a new model.

We don't have to write the binary cross entropy loss since it is already written for us (criterion).

Also instead of writing the optimization process which in our case was gradient descent (W[n+1] = w[n]-$\mu$dL/dW) we can use a pre-built optimizer (SGD).

In [ ]:
model = single_layer(90,1)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(),lr = 0.05)

There are a few pre-built training functions but usually the training function is written by hand. This function is similar to the one you wrote in this assignment only we now can use the pre-built tensor functions. Make sure you understood all the differences between the two:

In [ ]:
import pdb
def train_model(model, criterion, optimizer, batch_size=100, max_iters=100):
  iter = 0
  cost_list = []
  acc_list  = []
  train_norm_xs_shuff = train_norm_xs
  train_ts_shuff = train_ts
  val_X_tensor = torch.tensor(val_norm_xs,dtype=torch.float32)
  while iter < max_iters:
    # shuffle the training set (there is code above for how to do this)
    reindex = np.random.permutation(len(train_norm_xs))
    train_norm_xs_shuff = train_norm_xs_shuff[reindex]
    train_ts_shuff = train_ts_shuff[reindex]

    for i in range(0, len(train_norm_xs), batch_size): # iterate over each minibatch
      # minibatch that we are working with:
      X = train_norm_xs_shuff[i:(i + batch_size)]
      t = train_ts_shuff[i:(i + batch_size), 0]

      # since len(train_norm_xs) does not divide batch_size evenly, we will skip over
      # the "last" minibatch
      if np.shape(X)[0] != batch_size:
        continue
      # change the numpy types into torches tensors
      X_tensor = torch.tensor(X,dtype=torch.float32)
      t_tensor = torch.tensor(t,dtype=torch.float32).unsqueeze(1) # the unsqueeze reshapes (N,) to (N,1)

      # a clean up step for PyTorch
      optimizer.zero_grad()
      # compute the prediction
      prediction = model(X_tensor)
      # compute the cost/loss
      loss = criterion(prediction,t_tensor)
      # calculate gradient(backpropegate)
      loss.backward()
      # update w and b(step)
      optimizer.step()
      # increment the iteration count
      iter += 1
      # compute and print the *validation* accuracy
      if (iter % 40 == 0):
        val_pred = model(val_X_tensor)
        val_acc = get_accuracy(val_pred,val_ts)
        acc_list.append(val_acc)

        print("Iter %d. [Val Acc %.1f%%]" % (
                iter, val_acc * 100))

      if iter >= max_iters:
        break


  return acc_list

We can now run the training process. You should get pretty similar results to the ones from the model you wrote. Make sure that the results are in the same range and if not fix your model and try again.

In [ ]:
acc_list = train_model(model,criterion,optimizer,100,500)

You can also try to change the model (add layers or change layers) change the optimizer or the hyperparameters and try to improve the validation accuracy. If you want a challenge you can try to reach a validation accuracy of 75%